# Gemini Drive notes → n8n HQ Tasks

Runtime → **Run all**. Sign in with a Save 5 Hours Google account (`@save5hours.ch`, `antubejar96@gmail.com`, `deevlylabs@gmail.com`, or `roman.cajka@gmail.com`).

This creates a real Google Doc (Drive API) and POSTs `{fileId, text}` to `/webhook/public-drive-doc`. That path does not need n8n Google userinfo. It also tries `/webhook/meeting-notes-drive` with `googleAccessToken` when the Colab token is present.

You do **not** need an n8n login or `WEBHOOK_SECRET`.

Keep `VERIFY_NOTES` in sync with `fixtures/drive-verify-notes.txt`.

In [ ]:
%pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib requests

from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaInMemoryUpload
import requests

N8N = "https://n8n-production-192e.up.railway.app"
PUBLIC_WEBHOOK = N8N + "/webhook/public-drive-doc"
DRIVE_WEBHOOK = N8N + "/webhook/meeting-notes-drive"
TITLE = "Gemini notes — Drive path verification (n8n)"
VERIFY_NOTES = (
    "Gemini notes — Drive path verification (n8n)\n\n"
    "Attendees: Antoine Bejarano Alvarez, Martin, Roman Cajka.\n\n"
    "Actions agreed:\n"
    "- Antoine will publish the Drive webhook runbook in HQ this week.\n"
    "- Martin will review HQ Tasks with Origin Meeting after the Drive file lands.\n"
    "- Roman will confirm the Meet Recordings folder URL on the Drive confirmation task.\n"
)

creds, _ = google.auth.default()
if creds and creds.expired and getattr(creds, "refresh_token", None):
    creds.refresh(Request())
token = getattr(creds, "token", None) or ""
if not token:
    raise SystemExit("No Google access token. Re-run after signing in.")

drive = build("drive", "v3", credentials=creds)
media = MediaInMemoryUpload(VERIFY_NOTES.encode("utf-8"), mimetype="text/plain", resumable=False)
created = drive.files().create(
    body={"name": TITLE, "mimeType": "application/vnd.google-apps.document"},
    media_body=media,
    fields="id,webViewLink",
    supportsAllDrives=True,
).execute()
file_id = created["id"]
file_url = created.get("webViewLink") or f"https://docs.google.com/document/d/{file_id}/edit"
payload = {
    "fileId": file_id,
    "url": file_url,
    "name": TITLE,
    "mimeType": "application/vnd.google-apps.document",
    "webViewLink": file_url,
    "text": VERIFY_NOTES,
}

public = requests.post(PUBLIC_WEBHOOK, json=payload, timeout=120)
print("public-drive-doc HTTP", public.status_code)
print(public.text[:500])

if token:
    drive_resp = requests.post(
        DRIVE_WEBHOOK,
        json={**payload, "googleAccessToken": token},
        timeout=120,
    )
    print("meeting-notes-drive HTTP", drive_resp.status_code)
    print(drive_resp.text[:500])

print("FILE_ID", file_id)
print("FILE_URL", file_url)
if public.status_code >= 300:
    public.raise_for_status()